#### Purpose: Ingest raw graduate student longitudinal data, flag and fix quality issues, construct metrics, and export a clean parquet file for reporting and modeling.

#### Git commit history for this file:
##### Commit 1: Skeleton and raw loading of data (done)
##### Commit 2: Standardize data - cleaning
##### Commit 3: Construct financial support and career metrics
##### Commit 4: add data-quality report export

In [19]:
# importing packages
import pandas as pd
import numpy as np
from pathlib import Path
import json

### PATHS to relative project root

In [20]:
RAW_PATH = Path("data/raw/grad_students_raw.csv")
CLEAN_PATH = Path("data/processed/grad_students_clean.parquet")
QA_PATH = Path("data/processed/data_quality_report.json")

### Step 1: Loading data

In [21]:
## using Parquet for efficiency and reduction of column type errors
def load_raw(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)  # reads everything as string first, decide types later
    print(f"[LOAD] {len(df):,} x {len(df.columns)} columns loaded from {path}")
    return df

### Step 2: Standardizing data

In [22]:
BOOL_COLUMNS = ["pell_eligible", "TA_status", "GSR_status", "first_gen", "international"]

def _to_bool(series: pd.Series) -> pd.Series:
    """Convert free-text yes/no column to nullable boolean ([d.BooleanDtype)."""
    mapping = {"yes": True,
               "no": False,
               "1": True,
               "0": False,
               "true": True,
               "false": False}
    cleaned = series.str.strip().str.lower().map(mapping)
    return cleaned.astype(pd.BooleanDtype())

def standardize_strings(df: pd.DataFrame) -> pd.DataFrame:
    """title-case free-text columns, stripping whitespace"""
    str_cols = ["program",
                "thesis_status",
                "employment_outcome",
                "employer_type",
                "job_title",
                "industry",
                "gender",
                "ethnicity",
                "academic_term"]
    for col in str_cols:
        df[col] = df[col].str.strip().str.title()
        df["last_name"] = df["last_name"].str.strip().str.upper() ## upcase last name first
        df["first_name"] = df["first_name"].str.strip().str.title()
        print(f"[STANDARDIZE] String columns cleaned")
        return df

def standardize_numerics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cast numeric columns; coerce unparseable strings to NaN so we know
    exactly where the gaps are rather than silently dropping rows.
    """
    numeric_cols = {
        "grant_amt":          float,
        "gpa":                float,
        "units_enrolled":     int,
        "stipend_amt":        float,
        "time_to_degree":     float,
        "salary_offer":       float,
        "months_to_employment": float,
        "year_in_program":    int,
        "term_year":          int,
    }
    for col, dtype in numeric_cols.items():
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if dtype == int:
            df[col] = df[col].astype("Int64")   # nullable integer
        else:
            df[col] = df[col].astype(float)
    print(f"[STANDARDIZE] Numeric columns cast")
    return df


def standardize_strings(df: pd.DataFrame) -> pd.DataFrame:
    """Title-case free-text columns; strip whitespace."""
    str_cols = ["program", "thesis_status", "employment_outcome",
                "employer_type", "job_title", "industry", "gender",
                "ethnicity", "academic_term"]
    for col in str_cols:
        df[col] = df[col].str.strip().str.title()
    df["last_name"]  = df["last_name"].str.strip().str.upper()
    df["first_name"] = df["first_name"].str.strip().str.title()
    print(f"[STANDARDIZE] String columns cleaned")
    return df
